# 06 Deployment: Dashboard

ขั้นตอนนี้สร้าง dashboard 1 หน้าเป็นไฟล์ HTML ที่เปิดได้โดยไม่ต้องต่ออินเทอร์เน็ต ตาม Scope หัวข้อ 18 ฉบับปรับตามผลขั้น 05

**หลักการ**
- **origin ล่าสุด:** ใช้ origin ล่าสุดที่ไม่เกินวันที่ดาวน์โหลดข้อมูล (as-of) เพื่อให้ตรงกับกฎ forecast origin วันที่ 15 ของ t+2
- **โมเดล:** ใช้โมเดลที่เลือกจาก validation ในขั้น 05 พร้อม alpha หรือ ARIMA order ที่ตรึงไว้ แล้ว fit ใหม่ด้วยแถว training ที่ target เผยแพร่แล้ว ณ origin นั้น
- **ช่วงความคลาดเคลื่อน:** ± MAE ของช่วง test ของโมเดลเดียวกันที่ horizon เดียวกัน
- **ความโปร่งใส:** แสดงผลการประเมินจริงคู่กับค่าทำนายทุกครั้ง รวมถึงข้อที่ว่าสัญญาณ High Risk ไม่ได้ดีกว่าการเตือนทุกเดือน
- **ผลรอบที่ 2:** แสดงเป็นตารางและติดป้ายเชิงสำรวจ

In [1]:
from __future__ import annotations
import hashlib, html, json, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'PROJECT_SCOPE.md').exists())
RAW_DIR, PROCESSED_DIR, INTERIM_DIR = ROOT / 'data/raw', ROOT / 'data/processed', ROOT / 'data/interim'
METRICS_DIR, DASH_DIR = ROOT / 'logs/metrics', ROOT / 'reports/dashboard'
DASH_DIR.mkdir(parents=True, exist_ok=True)
GROUPS = ['318', '402']
GROUP_TH = {'318': 'ปุ๋ย และยากำจัดศัตรูพืช (318)', '402': 'นมและผลิตภัณฑ์นม (402)'}
GROUP_COLOR = {'318': '#2a78d6', '402': '#1baf7a'}
INK, MUTED, GRID = '#2b2b2b', '#6b6b66', '#e6e5df'
TH_MONTHS = ['ม.ค.', 'ก.พ.', 'มี.ค.', 'เม.ย.', 'พ.ค.', 'มิ.ย.', 'ก.ค.', 'ส.ค.', 'ก.ย.', 'ต.ค.', 'พ.ย.', 'ธ.ค.']
def th_month(ts): ts = pd.Timestamp(ts); return f'{TH_MONTHS[ts.month - 1]} {ts.year + 543}'
def th_date(ts): ts = pd.Timestamp(ts); return f'{ts.day} {th_month(ts)}'

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()

manifest = pd.read_csv(METRICS_DIR / '03_processed_manifest.csv')
assert sha256(ROOT / manifest.loc[0, 'file']) == manifest.loc[0, 'sha256']
dates = ['month_ce', 'forecast_origin', 'target_month', 'target_available_date']
modeling = pd.read_csv(ROOT / manifest.loc[0, 'file'], dtype={'commodity_code': str}, parse_dates=dates)
latest_features = pd.read_csv(PROCESSED_DIR / 'latest_features.csv', dtype={'commodity_code': str},
                              parse_dates=['month_ce', 'forecast_origin', 'fx_lead_t1_full_max_date', 'brent_lead_t1_full_max_date'])
FEATURE_SETS = {r.feature_set: r.features.split('|') for r in
                pd.read_csv(METRICS_DIR / '03_feature_sets.csv', dtype={'feature_set': str}).itertuples()}
best = pd.read_csv(METRICS_DIR / '05_best_model_by_validation.csv', dtype={'commodity_code': str})
test_metrics = pd.read_csv(METRICS_DIR / '05_test_metrics.csv', dtype={'commodity_code': str})
rq4 = pd.read_csv(METRICS_DIR / '05_rq4_horizon_skill.csv', dtype={'commodity_code': str})
alerts = pd.read_csv(METRICS_DIR / '05_alert_metrics.csv', dtype={'commodity_code': str})
dm_main = pd.read_csv(METRICS_DIR / '05_dm_main_comparisons.csv', dtype={'commodity_code': str})
iter2 = pd.read_csv(METRICS_DIR / '05b_results.csv', dtype={'commodity_code': str})
passthrough = pd.read_csv(METRICS_DIR / '02_passthrough_summary.csv')
alphas = pd.read_csv(METRICS_DIR / '04_ridge_alpha_selection.csv', dtype={'commodity_code': str, 'feature_set': str})
arima_grid = pd.read_csv(METRICS_DIR / '04_arima_order_selection.csv', dtype={'commodity_code': str})
pred04 = pd.read_csv(METRICS_DIR / '04_predictions.csv', dtype={'commodity_code': str, 'feature_set': str}, parse_dates=['month_ce'])
pred04['series'] = pred04.model + '/' + pred04.feature_set
external = pd.read_csv(INTERIM_DIR / 'external_monthly_aggregates.csv', parse_dates=['month_ce'])

sources = json.loads((RAW_DIR / 'sources.json').read_text(encoding='utf-8'))['sources']
imi_source = next(x for x in reversed(sources) if x['source_id'].startswith('imi-monthly-'))
assert sha256(RAW_DIR / imi_source['file']) == imi_source['sha256']
raw = pd.read_csv(RAW_DIR / imi_source['file'], dtype={'commodityCode': str})
raw['commodityCode'] = raw.commodityCode.str.zfill(3)
raw['month_ce'] = pd.to_datetime(dict(year=raw.year - 543, month=raw.month, day=1))
imi_series = {g: raw[raw.commodityCode == g].set_index('month_ce')['index'].sort_index().asfreq('MS') for g in GROUPS}
AS_OF = pd.Timestamp(imi_source['downloaded_at_utc'][:10])
print(f'as-of (วันที่ดาวน์โหลดข้อมูล): {AS_OF.date()}; plotly {plotly.__version__}')

as-of (วันที่ดาวน์โหลดข้อมูล): 2026-10-05; plotly 7.1.0


## 1. Origin ล่าสุดและฟังก์ชันพยากรณ์

เลือกเดือนฐาน t ล่าสุดที่ forecast origin ไม่เกินวันที่ as-of แล้วตรวจว่าข้อมูลภายนอกเดือน t+1 ครบก่อน origin

ฟังก์ชันพยากรณ์ใช้กฎเดียวกับขั้น 04 จึงตรวจความถูกต้องได้โดยคำนวณค่าทำนายของ origin ในช่วง test ซ้ำ แล้วเทียบกับ `04_predictions.csv` ต้องได้ค่าเท่ากัน

In [2]:
eligible = latest_features[latest_features.forecast_origin <= AS_OF]
T = eligible.month_ce.max()
ORIGIN = T + pd.offsets.MonthBegin(2) + pd.Timedelta(days=14)
latest_rows = eligible[eligible.month_ce == T].set_index('commodity_code')
assert (latest_rows.fx_lead_t1_full_max_date < ORIGIN).all() and (latest_rows.brent_lead_t1_full_max_date < ORIGIN).all()
assert all(imi_series[g].index.max() >= T for g in GROUPS)
print(f'เดือนฐาน t = {T:%Y-%m}; forecast origin = {ORIGIN.date()}; ข้อมูลภายนอกเดือน t+1 ถึง {latest_rows.fx_lead_t1_full_max_date.max().date()}')

BEST = {(r.commodity_code, int(r.horizon)): r.series for r in best.itertuples()}
arima_sel = arima_grid[arima_grid.converged].sort_values('aic').groupby('commodity_code').head(1).set_index('commodity_code')

def train_rows(g, h, origin):
    panel = modeling[(modeling.commodity_code == g) & (modeling.horizon == h)]
    train = panel[panel.target_available_date <= origin]
    assert train.month_ce.max() <= origin - pd.DateOffset(months=h + 2)
    return train

def forecast(g, h, t, features_row, origin):
    series = BEST[(g, h)]
    model, fs = series.split('/')
    train = train_rows(g, h, origin)
    out = {'series': series, 'train_n': len(train), 'threshold': float(np.quantile(train.target_y_pct, 0.8))}
    if model == 'ridge':
        feats = FEATURE_SETS[fs]
        alpha = float(alphas[(alphas.commodity_code == g) & (alphas.horizon == h) & (alphas.feature_set == fs) & alphas.selected].alpha.iloc[0])
        scaler = StandardScaler().fit(train[feats])
        ridge = Ridge(alpha=alpha).fit(scaler.transform(train[feats]), train.target_y_pct)
        x = scaler.transform(features_row[feats].to_frame().T.astype(float))
        out.update(y_pred=float(ridge.predict(x)[0]), alpha=alpha,
                   coefficients=pd.Series(ridge.coef_, index=feats))
    elif model == 'arima':
        sel = arima_sel.loc[g]
        log_imi = np.log(imi_series[g][:t])
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            result = ARIMA(log_imi, order=(int(sel.p), 1, int(sel.q)), trend=sel.trend).fit()
        out.update(y_pred=float(100 * (np.asarray(result.forecast(steps=h))[-1] - log_imi.iloc[-1])),
                   order=f'ARIMA({int(sel.p)},1,{int(sel.q)})')
    else:
        raise ValueError(series)
    return out

# ตรวจ: คำนวณค่าทำนายของ origin สุดท้ายในช่วง test ซ้ำ แล้วต้องเท่ากับผลขั้น 04
checks = []
for g in GROUPS:
    for h in range(2, 7):
        row = modeling[(modeling.commodity_code == g) & (modeling.horizon == h)].sort_values('month_ce').iloc[-1]
        again = forecast(g, h, row.month_ce, row, row.forecast_origin)['y_pred']
        stored = pred04[(pred04.commodity_code == g) & (pred04.horizon == h) & (pred04.series == BEST[(g, h)])
                        & (pred04.month_ce == row.month_ce)].y_pred.iloc[0]
        checks.append({'group': g, 'h': h, 't': row.month_ce.date(), 'series': BEST[(g, h)], 'recomputed': again, 'stored_04': stored})
checks = pd.DataFrame(checks)
assert np.allclose(checks.recomputed, checks.stored_04, atol=1e-6), checks
display(checks.round(4))
print('ฟังก์ชันพยากรณ์ของ dashboard ให้ค่าเท่ากับขั้น 04 ทุก horizon')

เดือนฐาน t = 2026-07; forecast origin = 2026-09-15; ข้อมูลภายนอกเดือน t+1 ถึง 2026-08-31


,group,h,t,series,recomputed,stored_04
0,318,2,2026-06-01,ridge/3,-5.8093,-5.8093
1,318,3,2026-05-01,ridge/3,1.0964,1.0964
2,318,4,2026-04-01,ridge/3,11.2966,11.2966
3,318,5,2026-03-01,ridge/3,13.4400,13.4400
4,318,6,2026-02-01,ridge/3,9.8063,9.8063
5,402,2,2026-06-01,arima/none,-0.9099,-0.9099
6,402,3,2026-05-01,ridge/3-cutoff24,-0.0616,-0.0616
7,402,4,2026-04-01,arima/none,-1.4479,-1.4479
8,402,5,2026-03-01,arima/none,-0.4727,-0.4727
9,402,6,2026-02-01,arima/none,-2.9901,-2.9901


ฟังก์ชันพยากรณ์ของ dashboard ให้ค่าเท่ากับขั้น 04 ทุก horizon


## 2. ค่าพยากรณ์ ณ origin ล่าสุด

ค่าพยากรณ์ y คือการเปลี่ยนแปลง log สะสมจากเดือน t (×100) แปลงเป็นระดับดัชนีด้วย IMI[t] × exp(y/100)
- ช่วงความคลาดเคลื่อนคือ ± MAE ช่วง test ของโมเดลเดียวกัน
- `passes` คือผลตามเกณฑ์ "ยังเตือนได้" ของ RQ4(ก) จากขั้น 05

In [3]:
rows, coefs = [], {}
for g in GROUPS:
    imi_t = float(imi_series[g][T])
    for h in range(2, 7):
        f = forecast(g, h, T, latest_rows.loc[g], ORIGIN)
        mae = float(test_metrics[(test_metrics.commodity_code == g) & (test_metrics.horizon == h) & (test_metrics.series == f['series'])].MAE.iloc[0])
        r4 = rq4[(rq4.commodity_code == g) & (rq4.horizon == h)].iloc[0]
        rows.append({'group': g, 'h': h, 'target_month': T + pd.DateOffset(months=h), 'series': f['series'],
                     'y_pred': f['y_pred'], 'mae_test': mae, 'imi_t': imi_t,
                     'index_pred': imi_t * np.exp(f['y_pred'] / 100),
                     'index_low': imi_t * np.exp((f['y_pred'] - mae) / 100), 'index_high': imi_t * np.exp((f['y_pred'] + mae) / 100),
                     'threshold': f['threshold'], 'high_risk': f['y_pred'] > f['threshold'],
                     'skill_test': r4.skill, 'p_test': r4.p_value, 'passes': bool(r4.passes), 'train_n': f['train_n']})
        if 'coefficients' in f and h == 3: coefs[g] = f['coefficients']
forecasts = pd.DataFrame(rows)
assert np.isfinite(forecasts[['y_pred', 'index_pred', 'index_low', 'index_high']]).all().all()
forecasts.to_csv(METRICS_DIR / '06_latest_forecasts.csv', index=False)
display(forecasts.assign(target_month=forecasts.target_month.dt.strftime('%Y-%m')).round(3))

,group,h,target_month,series,y_pred,mae_test,imi_t,index_pred,index_low,index_high,threshold,high_risk,skill_test,p_test,passes,train_n
0,318,2,2026-09,ridge/3,-4.327,4.630,95.1,91.073,86.953,95.389,2.224,False,0.052,0.413,False,305
1,318,3,2026-10,ridge/3,-5.508,6.367,95.1,90.004,84.452,95.921,3.178,False,0.017,0.798,False,304
2,318,4,2026-11,ridge/3,-7.361,7.651,95.1,88.351,81.844,95.376,4.212,False,-0.005,0.954,False,303
3,318,5,2026-12,ridge/3,-5.785,8.651,95.1,89.755,82.316,97.866,5.074,False,-0.020,0.770,False,302
4,318,6,2027-01,ridge/3,-5.925,10.101,95.1,89.629,81.017,99.156,5.932,False,-0.052,0.487,False,301
5,402,2,2026-09,arima/none,-1.020,1.488,134.2,132.838,130.875,134.830,2.108,False,0.183,0.022,True,305
6,402,3,2026-10,ridge/3-cutoff24,-0.635,2.199,134.2,133.350,130.449,136.315,3.095,False,0.134,0.063,True,304
7,402,4,2026-11,arima/none,-1.777,2.689,134.2,131.837,128.339,135.429,4.413,False,0.185,0.056,True,303
8,402,5,2026-12,arima/none,-2.120,3.248,134.2,131.385,127.186,135.723,5.193,False,0.186,0.083,True,302
9,402,6,2027-01,arima/none,-2.338,3.841,134.2,131.099,126.159,136.233,6.203,False,0.168,0.146,False,301


## 3. สร้าง dashboard HTML

ไฟล์ HTML ฝัง plotly.js ไว้ในไฟล์ จึงเปิดได้โดยไม่ต้องต่ออินเทอร์เน็ต ตัวอักษรใช้ IBM Plex Sans Thai จาก Google Fonts ถ้าออฟไลน์จะใช้ฟอนต์ของระบบแทน

In [4]:
FONT = "'IBM Plex Sans Thai', 'Sarabun', 'Noto Sans Thai', system-ui, sans-serif"
CONFIG = {'displayModeBar': False, 'responsive': True}

def base_layout(fig, height):
    fig.update_layout(height=height, margin=dict(l=56, r=20, t=16, b=44), paper_bgcolor='white', plot_bgcolor='white',
                      font=dict(family=FONT, size=13, color=INK), hoverlabel=dict(font_family=FONT),
                      legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0, font=dict(size=12)))
    fig.update_xaxes(showgrid=False, linecolor=GRID, ticks='outside', tickcolor=GRID)
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=GRID)
    return fig

def forecast_chart(g):
    hist = imi_series[g]['2015-01-01':T]
    fc = forecasts[forecasts.group == g]
    color = GROUP_COLOR[g]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist.index, y=hist.values, mode='lines', name='IMI จริง', line=dict(color=INK, width=2),
                             hovertemplate='%{x|%b %Y}<br>IMI %{y:.1f}<extra></extra>'))
    x_band = [T, *fc.target_month, *fc.target_month[::-1], T]
    y_band = [hist.iloc[-1], *fc.index_high, *fc.index_low[::-1], hist.iloc[-1]]
    fig.add_trace(go.Scatter(x=x_band, y=y_band, mode='lines', fill='toself', fillcolor='rgba(42,120,214,0.12)' if g == '318' else 'rgba(27,175,122,0.14)',
                             line=dict(width=0), hoverinfo='skip', name='± MAE ช่วง test'))
    fig.add_trace(go.Scatter(x=[T, *fc.target_month], y=[hist.iloc[-1], *fc.index_pred], mode='lines+markers', name='ค่าพยากรณ์',
                             line=dict(color=color, width=2, dash='dot'),
                             marker=dict(size=9, color=[color if p else 'white' for p in [True, *fc.passes]],
                                         line=dict(color=color, width=2)),
                             customdata=np.array([[0, 0, 0, '-', '-']] + [[r.y_pred, r.mae_test, r.h, r.series, 'ผ่าน' if r.passes else 'ไม่ผ่าน']
                                                                       for r in fc.itertuples()], dtype=object),
                             hovertemplate='%{x|%b %Y}<br>ดัชนีคาด %{y:.1f}<br>เปลี่ยนจากเดือน t %{customdata[0]:+.1f}% (± %{customdata[1]:.1f})'
                                           '<br>h=%{customdata[2]} · %{customdata[3]}<br>เกณฑ์ยังเตือนได้: %{customdata[4]}<extra></extra>'))
    fig.add_vline(x=T, line=dict(color=MUTED, width=1, dash='dot'))
    fig.add_annotation(x=T, y=1, yref='paper', text=f'เดือนฐาน t ({th_month(T)})', showarrow=False, xanchor='right',
                       yanchor='top', font=dict(size=11, color=MUTED))
    fig.update_yaxes(title_text='ดัชนี (ปี 2555 = 100)')
    return base_layout(fig, 380)

def coef_chart(g):
    if g not in coefs: return None
    c = coefs[g].reindex(coefs[g].abs().sort_values(ascending=True).index).tail(10)
    fig = go.Figure(go.Bar(x=c.values, y=c.index, orientation='h', marker=dict(color=GROUP_COLOR[g]),
                           hovertemplate='%{y}<br>coefficient %{x:+.3f}<extra></extra>'))
    fig.add_vline(x=0, line=dict(color=MUTED, width=1))
    fig.update_xaxes(title_text='coefficient (feature ปรับมาตรฐานแล้ว)', title_font=dict(size=12))
    fig.update_layout(showlegend=False)
    return base_layout(fig, 360)

def drivers_chart():
    ext = external[(external.month_ce >= T - pd.DateOffset(months=59)) & (external.month_ce <= T + pd.DateOffset(months=1))]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                        subplot_titles=('อัตราแลกเปลี่ยน (บาทต่อดอลลาร์, ค่าเฉลี่ยรายเดือน)', 'ราคาน้ำมัน Brent (ดอลลาร์ต่อบาร์เรล, ค่าเฉลี่ยรายเดือน)'))
    for i, (sid, unit) in enumerate((('DEXTHUS', 'บาท/ดอลลาร์'), ('DCOILBRENTEU', 'ดอลลาร์/บาร์เรล')), start=1):
        s = ext[ext.series == sid]
        fig.add_trace(go.Scatter(x=s.month_ce, y=s.full_mean, mode='lines', line=dict(color=INK if i == 1 else '#eb6834', width=2),
                                 name=sid, hovertemplate=f'%{{x|%b %Y}}<br>%{{y:.2f}} {unit}<extra></extra>'), row=i, col=1)
    fig.update_annotations(font=dict(size=12, color=MUTED), x=0, xanchor='left')
    fig.update_layout(showlegend=False)
    return base_layout(fig, 360)

figs = {g: {'forecast': forecast_chart(g), 'coef': coef_chart(g)} for g in GROUPS}
drivers_html = drivers_chart().to_html(full_html=False, include_plotlyjs=False, config=CONFIG)
print('สร้างกราฟครบ:', {g: [k for k, v in d.items() if v is not None] for g, d in figs.items()})

สร้างกราฟครบ: {'318': ['forecast', 'coef'], '402': ['forecast', 'coef']}


In [5]:
def esc(x): return html.escape(str(x))
def pct(x, d=1): return f'{100 * x:+.{d}f}%'
def pval(p): return '< 0.001' if p < 0.001 else f'{p:.3f}'

def kpi_tiles(g):
    fc3 = forecasts[(forecasts.group == g) & (forecasts.h == 3)].iloc[0]
    arrow = '▲ ขึ้น' if fc3.y_pred > 0 else '▼ ลง'
    hr = alerts[(alerts.commodity_code == g) & (alerts.series == fc3.series)].iloc[0]
    always = alerts[(alerts.commodity_code == g) & (alerts.series == 'always_alert')].iloc[0]
    status = 'High Risk' if fc3.high_risk else 'Normal'
    prec = '–' if pd.isna(hr.precision) else f'{hr.precision:.2f}'
    return f"""
    <div class="kpis">
      <div class="tile"><div class="label">IMI ล่าสุดที่ใช้ (เดือนฐาน t)</div><div class="value">{fc3.imi_t:.1f}</div><div class="sub">{th_month(T)} · ปีฐาน 2555 = 100</div></div>
      <div class="tile"><div class="label">คาดการณ์ {th_month(fc3.target_month)} (h=3)</div><div class="value">{fc3.y_pred:+.1f}%</div><div class="sub">ดัชนีคาด {fc3.index_pred:.1f} · ช่วง {fc3.index_low:.1f}–{fc3.index_high:.1f} (± MAE {fc3.mae_test:.1f})</div></div>
      <div class="tile"><div class="label">ทิศทางที่คาด (h=3)</div><div class="value">{arrow}</div><div class="sub">โมเดล {esc(fc3.series)} · เลือกจากช่วง validation</div></div>
      <div class="tile warn"><div class="label">สถานะ High Risk (h=3)</div><div class="value">{status}</div><div class="sub">threshold {fc3.threshold:.1f}% · ⚠ ช่วง test: precision {prec}, recall {hr.recall:.2f} เทียบเตือนทุกเดือน precision {always.precision:.2f} — สัญญาณนี้ยังไม่ดีกว่าการเตือนทุกเดือน</div></div>
    </div>"""

def eval_table(g):
    fc = forecasts[forecasts.group == g]
    i2 = iter2[iter2.commodity_code == g].set_index('horizon')
    body = ''.join(
        f"<tr><td>h={r.h} · {th_month(r.target_month)}</td><td>{esc(r.series)}</td><td class='num'>{r.y_pred:+.1f}%</td>"
        f"<td class='num'>{pct(r.skill_test)}</td><td class='num'>{pval(r.p_test)}</td>"
        f"<td>{'<span class=\"ok\">● ผ่าน</span>' if r.passes else '<span class=\"no\">○ ไม่ผ่าน</span>'}</td>"
        f"<td class='num muted'>{pct(i2.loc[r.h, 'skill_vs_no_change'])} (p {pval(i2.loc[r.h, 'p_vs_no_change'])})</td></tr>"
        for r in fc.itertuples())
    return f"""<table><thead><tr><th>เดือนเป้าหมาย</th><th>โมเดล</th><th class='num'>ค่าคาด y</th>
      <th class='num'>skill ช่วง test<br><span class='muted'>เทียบ No-change</span></th><th class='num'>DM-HLN p</th>
      <th>เกณฑ์ยังเตือนได้<br><span class='muted'>skill&gt;0 และ p&lt;0.10</span></th>
      <th class='num'>รอบที่ 2 (เชิงสำรวจ)<br><span class='muted'>+ราคาวัตถุดิบโลก</span></th></tr></thead><tbody>{body}</tbody></table>"""

def findings(g):
    c1 = dm_main[(dm_main.commodity_code == g) & dm_main.comparison.str.startswith('C1')].iloc[0]
    c2 = dm_main[(dm_main.commodity_code == g) & dm_main.comparison.str.startswith('C2')].iloc[0]
    if g == '402':
        pt = passthrough[passthrough.pair.str.contains('CPI')].iloc[0]
        pass_txt = (f"ในข้อมูลก่อนปี 2562 CPI นมเปลี่ยนตามราคานำเข้าในรูปบาทย้อนหลัง ค่าประมาณ median lag <b>{int(pt.median_lag_months)} เดือน</b> "
                    f"และส่งผ่านเพียงบางส่วน (ราคานำเข้า +10% ≈ CPI นม +{10 * pt.cumulative_12m:.1f}% ใน 12 เดือน) เป็นความสัมพันธ์ย้อนหลัง ยังไม่ได้ทดสอบว่าเพิ่มระยะเตือนราคาที่ SME จ่ายได้จริง")
        use_txt = 'ผลเชิงสำรวจผ่านเกณฑ์ที่ h=2–5 โดย h นับจากเดือนฐาน t: h=2 เป็น nowcast ของเดือนที่ออก forecast ส่วน h=3–5 เป็นเดือนเป้าหมายถัดไปถึงอีก 3 เดือน ใช้ประกอบการคุยกับ supplier หรือทบทวนงบ ความแม่นมาจากประวัติ IMI เป็นหลัก'
    else:
        pass_txt = 'ไม่พบหลักฐานว่าราคาในประเทศตามหลังราคานำเข้า (ใช้ PPI 3201 ซึ่งรวมพลาสติกและมีข้อมูลสั้น) ยังยืนยันระยะเตือนเพิ่มเติมจาก pass-through ไม่ได้ โดยเดือนเป้าหมายอยู่ห่างจากเดือนที่ออก forecast h−2 เดือน'
        use_txt = 'ใช้ประกอบการดูทิศทางเท่านั้น ค่าเงินและน้ำมันไม่ช่วยพยากรณ์ขนาดการเปลี่ยนแปลงอย่างมีนัยสำคัญ ราคาปุ๋ยโลก (ยูเรีย/DAP) ในรอบที่ 2 มีแนวโน้มช่วยที่ h=2–3 แต่เป็นผลเชิงสำรวจ'
    return f"""<ul class="findings">
      <li><b>ใช้อย่างไร:</b> {use_txt}</li>
      <li><b>ข้อมูลภายนอกช่วยไหม (h=3, ชุด 3 เทียบชุด 1):</b> MAE {c1.MAE_1:.2f} เทียบ {c1.MAE_2:.2f} (Holm p = {c1.p_holm:.2f}) ไม่มีนัยสำคัญ</li>
      <li><b>เทียบ No-change (h=3):</b> {esc(c2.model_1)} MAE {c2.MAE_1:.2f} เทียบ {c2.MAE_2:.2f} (Holm p = {c2.p_holm:.2f})</li>
      <li><b>ราคาในประเทศตามหลังกี่เดือน:</b> {pass_txt}</li></ul>"""

def section(g, active):
    f = figs[g]
    coef_html = (f['coef'].to_html(full_html=False, include_plotlyjs=False, config=CONFIG) if f['coef'] is not None
                 else '<p class="muted">โมเดลที่เลือกสำหรับ h=3 ไม่ใช่ Ridge จึงไม่มี coefficient</p>')
    fc3 = forecasts[(forecasts.group == g) & (forecasts.h == 3)].iloc[0]
    return f"""
  <section id="g{g}" class="panel" {'hidden' if not active else ''}>
    {kpi_tiles(g)}
    <div class="card"><h3>IMI จริงและค่าพยากรณ์ h=2–6</h3>
      <p class="muted">จุดทึบ = horizon ที่ผ่านเกณฑ์เชิงสำรวจในช่วง test · h นับจากเดือนฐาน t; h=2 เป็น nowcast · แถบ = ± MAE ช่วง test ไม่ใช่ช่วงความเชื่อมั่น · origin {th_date(ORIGIN)}</p>
      {f['forecast'].to_html(full_html=False, include_plotlyjs=False, config=CONFIG)}</div>
    <div class="card"><h3>ผลการประเมินช่วง test (เดือนฐาน ม.ค. 2562 เป็นต้นไป)</h3>{eval_table(g)}</div>
    <div class="grid2">
      <div class="card"><h3>สรุปสำหรับผู้ใช้</h3>{findings(g)}</div>
      <div class="card"><h3>ปัจจัยที่โมเดล h=3 ให้น้ำหนัก ({esc(fc3.series)})</h3>{coef_html}</div>
    </div>
  </section>"""

title = 'IMI Early Warning — ปุ๋ยและนมนำเข้า'
page = f"""<!doctype html>
<html lang="th"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>IMI Early Warning</title>
<link rel="preconnect" href="https://fonts.googleapis.com"><link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Sans+Thai:wght@400;600&display=swap" rel="stylesheet">
<style>
:root {{ --bg:#f7f6f2; --surface:#ffffff; --ink:{INK}; --muted:{MUTED}; --line:{GRID}; --warn:#b26b00; --ok:#1a7f4b; }}
* {{ box-sizing:border-box; }}
body {{ margin:0; background:var(--bg); color:var(--ink); font-family:{FONT}; font-size:15px; line-height:1.55; }}
.wrap {{ max-width:1180px; margin:0 auto; padding:24px 16px 48px; }}
header h1 {{ margin:0 0 4px; font-size:24px; }} header p {{ margin:0; color:var(--muted); }}
.banner {{ margin:16px 0; padding:12px 14px; background:#fff7e6; border:1px solid #f0d9a8; border-radius:8px; }}
nav {{ display:flex; gap:8px; margin:16px 0; flex-wrap:wrap; }}
nav button {{ font:inherit; padding:8px 14px; border-radius:999px; border:1px solid var(--line); background:var(--surface); cursor:pointer; }}
nav button[aria-selected="true"] {{ background:var(--ink); color:#fff; border-color:var(--ink); }}
.kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(220px,1fr)); gap:12px; margin-bottom:12px; }}
.tile, .card {{ background:var(--surface); border:1px solid var(--line); border-radius:10px; padding:14px 16px; }}
.tile .label {{ color:var(--muted); font-size:13px; }} .tile .value {{ font-size:26px; font-weight:600; margin:2px 0; }}
.tile .sub {{ color:var(--muted); font-size:12.5px; }} .tile.warn {{ border-color:#f0d9a8; }}
.card {{ margin-bottom:12px; overflow-x:auto; }} .card h3 {{ margin:0 0 6px; font-size:16px; }}
.grid2 {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(340px,1fr)); gap:12px; }}
table {{ border-collapse:collapse; width:100%; font-size:14px; }} th, td {{ padding:7px 8px; border-bottom:1px solid var(--line); text-align:left; vertical-align:top; }}
th {{ font-weight:600; font-size:13px; }} .num {{ text-align:right; font-variant-numeric:tabular-nums; }}
.muted {{ color:var(--muted); font-weight:400; }} .ok {{ color:var(--ok); }} .no {{ color:var(--muted); }}
.findings {{ margin:0; padding-left:18px; }} .findings li {{ margin-bottom:8px; }}
footer {{ margin-top:20px; color:var(--muted); font-size:13px; }}
</style>
<script>{plotly.offline.get_plotlyjs()}</script>
</head><body><div class="wrap">
<header><h1>{esc(title)}</h1>
<p>ข้อมูล ณ {th_date(AS_OF)} · เดือนฐาน t = {th_month(T)} · forecast origin {th_date(ORIGIN)} · อัปเดตถัดไปเมื่อ IMI {th_month(T + pd.DateOffset(months=1))} พร้อมใช้ ({th_date(ORIGIN + pd.DateOffset(months=1))})</p></header>
<div class="banner"><b>อ่านก่อนใช้:</b> ค่าทำนายเป็นสัญญาณระดับกลุ่มสินค้าของดัชนีราคานำเข้า (C.I.F. ในรูปดอลลาร์สหรัฐ) ไม่ใช่ต้นทุนของบริษัทใด และไม่ใช่คำแนะนำให้ซื้อหรือไม่ซื้อ
ในช่วงทดสอบ ข้อมูลค่าเงินและน้ำมันไม่ได้ทำให้พยากรณ์แม่นขึ้นอย่างมีนัยสำคัญ และสัญญาณ High Risk ไม่ได้ดีกว่าการเตือนทุกเดือน ผลรอบที่ 2 เป็นผลเชิงสำรวจหลังเห็นผลทดสอบแล้ว</div>
<nav role="tablist">{''.join(f'<button role="tab" data-target="g{g}" aria-selected="{str(i == 0).lower()}">{esc(GROUP_TH[g])}</button>' for i, g in enumerate(GROUPS))}</nav>
{''.join(section(g, i == 0) for i, g in enumerate(GROUPS))}
<div class="card"><h3>ข้อมูลภายนอก 5 ปีล่าสุด (ถึงเดือน t+1)</h3>{drivers_html}</div>
<footer>แหล่งข้อมูล: สนค. กระทรวงพาณิชย์ (IMI, CPI, PPI; Open Data Common) · FRED: DEXTHUS (Federal Reserve H.10), DCOILBRENTEU (U.S. EIA) · World Bank Pink Sheet (CC BY 4.0) · FAO Food Price Index ·
วิธีการและข้อจำกัดทั้งหมดอยู่ใน PROJECT_SCOPE.md และ notebook 01–06 · สร้างเมื่อ {datetime.now():%Y-%m-%d %H:%M}</footer>
</div>
<script>
function showTab(id) {{
  document.querySelectorAll('nav button').forEach(b => b.setAttribute('aria-selected', b.dataset.target === id));
  document.querySelectorAll('.panel').forEach(p => p.hidden = p.id !== id);
  document.querySelectorAll('#' + id + ' .plotly-graph-div').forEach(d => Plotly.Plots.resize(d));
}}
document.querySelectorAll('nav button').forEach(btn => btn.addEventListener('click', () => {{
  showTab(btn.dataset.target); history.replaceState(null, '', '#' + btn.dataset.target);
}}));
if (document.getElementById(location.hash.slice(1))) showTab(location.hash.slice(1));
</script></body></html>"""
out_path = DASH_DIR / 'imi_dashboard.html'
out_path.write_text(page, encoding='utf-8')
size_mb = out_path.stat().st_size / 1e6
assert all(f'id="g{g}"' in page for g in GROUPS) and size_mb < 16
print(f'บันทึก {out_path.relative_to(ROOT)} ({size_mb:.1f} MB)')

บันทึก reports/dashboard/imi_dashboard.html (4.9 MB)


## 4. Checkpoint

เงื่อนไขผ่านตาม Scope หัวข้อ 15.3: แสดงผลได้ถูกต้องกับข้อมูลเดือนล่าสุด
- origin ไม่เกินวันที่ as-of
- ข้อมูลภายนอกเดือน t+1 ครบก่อน origin
- ฟังก์ชันพยากรณ์ให้ค่าเท่ากับขั้น 04
- ค่าพยากรณ์ครบทุกกลุ่มและ horizon
- ไฟล์ HTML มีทั้งสองกลุ่ม

In [6]:
assert ORIGIN <= AS_OF
assert len(forecasts) == len(GROUPS) * 5
assert (forecasts.train_n > 0).all()
print('CHECKPOINT PASSED: dashboard ใช้ข้อมูลถึงเดือนฐาน', f'{T:%Y-%m}', 'และค่าพยากรณ์ตรวจสอบกับขั้น 04 แล้ว')

log_path = ROOT / 'logs/process_log.md'
log = log_path.read_text(encoding='utf-8')
marker = 'ขั้นตอน: 06 Deployment'
if marker not in log:
    fc3 = forecasts[forecasts.h == 3].set_index('group')
    entry = f"""

## [2026-10-05] {marker}
**สิ่งที่ทำ:** สร้าง dashboard HTML 1 หน้า (reports/dashboard/imi_dashboard.html) ที่ฝัง plotly.js ไว้ในไฟล์ แสดงค่าพยากรณ์ h=2–6 ของกลุ่มปุ๋ยและนม ณ origin ล่าสุด พร้อมผลการประเมินช่วง test ผลรอบที่ 2 (ติดป้ายเชิงสำรวจ) และข้อมูลภายนอก

**ผลที่ได้:**
- as-of {AS_OF.date()} เดือนฐาน t = {T:%Y-%m} forecast origin {ORIGIN.date()}
- ค่าพยากรณ์ h=3: ปุ๋ย {fc3.loc['318', 'y_pred']:+.2f}% ({fc3.loc['318', 'series']}); นม {fc3.loc['402', 'y_pred']:+.2f}% ({fc3.loc['402', 'series']})
- ฟังก์ชันพยากรณ์ของ dashboard ให้ค่าเท่ากับขั้น 04 ทุก horizon เมื่อคำนวณซ้ำที่ origin สุดท้ายของช่วง test
- ไฟล์ขนาด {size_mb:.1f} MB และบันทึกค่าพยากรณ์ไว้ที่ logs/metrics/06_latest_forecasts.csv

**สิ่งที่พบ / ปัญหา:** IMI เดือน {T + pd.DateOffset(months=1):%Y-%m} มีใน API แล้ว แต่ตามกฎ forecast origin จะใช้ได้ตั้งแต่วันที่ {(ORIGIN + pd.DateOffset(months=1)).date()} dashboard จึงยังไม่ใช้เดือนนั้น

**การตัดสินใจและเหตุผล:** แสดงคำเตือนว่าสัญญาณ High Risk ในช่วง test ไม่ได้ดีกว่าการเตือนทุกเดือน และแสดง skill กับ p-value คู่กับค่าทำนายทุก horizon เพื่อไม่ให้ผู้ใช้เชื่อค่าทำนายเกินหลักฐาน

**ขั้นต่อไป:** เขียนรายงาน CRISP-DM จาก process log และเตรียมนำเสนอ
"""
    log_path.write_text(log.rstrip() + entry + '\n', encoding='utf-8')
    print('บันทึกผลลง logs/process_log.md แล้ว')
else:
    print('มีรายการ 06 Deployment ใน process log แล้ว จึงไม่บันทึกซ้ำ')

CHECKPOINT PASSED: dashboard ใช้ข้อมูลถึงเดือนฐาน 2026-07 และค่าพยากรณ์ตรวจสอบกับขั้น 04 แล้ว
มีรายการ 06 Deployment ใน process log แล้ว จึงไม่บันทึกซ้ำ
